# EDA — Home Credit Default Risk (Фаза 1)

Анализ собранной client-level витрины фичей и сырых заявок: баланс классов,
пропуски, распределения ключевых признаков, мультиколлинеарность и аномалии.
Выводы — в последней секции и в `docs/eda.md`.

Перед запуском: `make download` (данные) и `make features` (витрина).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from pd_scoring.eda import (
    days_employed_anomaly,
    missingness,
    target_rate,
    top_correlations,
)

DATA = Path("data")
mart = pl.read_parquet(DATA / "processed" / "mart.parquet")
app = pl.read_csv(DATA / "raw" / "application_train.csv", infer_schema_length=20000)
print("mart:", mart.shape, "| application_train:", app.shape)

## 1. Баланс классов (доля дефолтов)

In [ ]:
tr = target_rate(mart)
print("Размечено заявок:", int(tr["n"]))
print("Дефолтов:", int(tr["positives"]), "=> rate {:.2%}".format(tr["rate"]))
plt.figure(figsize=(4, 3))
plt.bar(
    ["no default", "default"],
    [tr["n"] - tr["positives"], tr["positives"]],
    color=["#4c78a8", "#e45756"],
)
plt.title("Class balance")
plt.tight_layout()
plt.show()

## 2. Пропуски (топ-20 колонок)

In [ ]:
miss = missingness(mart)
miss_top = miss.head(20)
with pl.Config(tbl_rows=20):
    print(miss_top)

## 3. Распределения ключевых признаков

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(mart["EXT_SOURCE_2"].drop_nulls().to_numpy(), bins=50, color="#4c78a8")
ax[0].set_title("EXT_SOURCE_2")
ax[1].hist((-app["DAYS_BIRTH"] / 365).to_numpy(), bins=50, color="#54a24b")
ax[1].set_title("Возраст, лет")
ratio = mart["CREDIT_INCOME_RATIO"].drop_nulls().to_numpy()
ax[2].hist(ratio, bins=50, range=(0, 20), color="#f58518")
ax[2].set_title("CREDIT / INCOME")
plt.tight_layout()
plt.show()

## 4. Мультиколлинеарность / связь с таргетом

In [ ]:
corr = top_correlations(mart, k=15)
corr_df = pl.DataFrame(
    {
        "feature": [c for c, _ in corr],
        "corr_with_target": [round(v, 4) for _, v in corr],
    }
)
print(corr_df)

## 5. Аномалия DAYS_EMPLOYED (365243)

In [ ]:
anom = days_employed_anomaly(app)
print(
    "DAYS_EMPLOYED == 365243:",
    int(anom["anomaly_count"]),
    "=> {:.2%} заявок".format(anom["anomaly_frac"]),
)
print("В витрине вынесено во флаг DAYS_EMPLOYED_ANOM, значение вычищено в null.")

## 6. Сохранить выводы в docs/eda.md

In [ ]:
n_features = len([c for c in mart.columns if c not in ("SK_ID_CURR", "TARGET", "is_train")])
lines = [
    "# EDA — выводы (Home Credit, Фаза 1)",
    "",
    f"- Витрина: {mart.height} клиентов x {mart.width} колонок ({n_features} фич).",
    f"- Баланс классов: дефолтов {int(tr['positives'])} из {int(tr['n'])} "
    f"({tr['rate']:.2%}) — сильный дисбаланс (~1:11), учитывать при обучении/метриках.",
    f"- Аномалия DAYS_EMPLOYED==365243: {int(anom['anomaly_count'])} "
    f"({anom['anomaly_frac']:.1%}) — заглушка неработающих; вынесена во флаг и вычищена в null.",
    "- EXT_SOURCE_1/2/3 — сильнейшие предикторы (макс. |corr| с таргетом), но с пропусками.",
    "",
    "## Топ-15 пропусков",
    "",
]
for row in missingness(mart).head(15).iter_rows(named=True):
    lines.append(f"- `{row['column']}`: {row['null_frac']:.1%}")
lines += ["", "## Топ-15 корреляций с TARGET", ""]
for c, v in corr:
    lines.append(f"- `{c}`: {v:+.3f}")
Path("docs").mkdir(exist_ok=True)
Path("docs/eda.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("written docs/eda.md")

## Выводы

- **Сильный дисбаланс классов** (~8% дефолтов): для обучения нужны class weights /
  PR-ориентированные метрики (PR-AUC, KS, Gini), не просто accuracy.
- **EXT_SOURCE_1/2/3** — самые предиктивные признаки, но с заметными пропусками →
  важна аккуратная обработка пропусков (в WOE — отдельная корзина).
- **Аномалия `DAYS_EMPLOYED==365243`** — техническая заглушка (неработающие): вычищена
  в null + флаг `DAYS_EMPLOYED_ANOM`, иначе исказила бы стаж и производные фичи.
- Денежные/возрастные распределения скошены (длинный хвост) → в Фазе 2 для GBDT ок,
  для scorecard поможет WOE-биннинг.
- Часть агрегатов бюро/карт сильно коррелируют между собой (мультиколлинеарность) —
  для логистической scorecard отбирать по IV/VIF, для GBDT не критично.